# Colab Dense Embedding & Indexing - LegalRAG

Notebook này tự động hóa 100% quá trình tạo **Dense Vector Index** cho dữ liệu Bộ luật Lao động 2019 với **cả 2 mô hình Embedding theo cấu hình nghiên cứu**:
1. **MiniLM-L12-v2**: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (384 chiều, gọn nhẹ, baseline)
2. **BGE-M3**: `BAAI/bge-m3` (1024 chiều, SOTA đa ngữ, hỗ trợ ngữ cảnh pháp lý sâu)

⚡ **Chỉ cần bấm: Runtime -> Run all (hoặc `Ctrl + F9`)** để chạy tự động trên GPU Colab.

In [ ]:
# 1. Kiểm tra GPU và môi trường
!nvidia-smi
import torch

print("\n" + "=" * 50)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> T4 GPU để tăng tốc.")
print("=" * 50)

In [ ]:
# 2. Clone mã nguồn LegalRAG và nạp thư viện
import os
import sys
from pathlib import Path

target_dir = Path("/content/LegalRAG")

# Kiểm tra thư mục src/legal_rag, nếu chưa có hoặc bị lỗi thì clone mới hoàn toàn
if not (target_dir / "src" / "legal_rag").exists():
    print("Mã nguồn chưa tồn tại hoặc bị thiếu. Đang clone mới...")
    !rm -rf /content/LegalRAG
    !git clone https://github.com/ngminhduc129/LegalRAG.git /content/LegalRAG
else:
    print("Cập nhật mã nguồn mới nhất...")
    !git -C /content/LegalRAG pull

%cd /content/LegalRAG

# Cài đặt các thư viện cần thiết
!pip install sentence-transformers underthesea rank-bm25 typer rich loguru pydantic pydantic-yaml --quiet

# Thêm src vào sys.path để import legal_rag trực tiếp
src_dir = str((target_dir / "src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import legal_rag
print(f"\n✓ Đã kết nối mã nguồn LegalRAG v{legal_rag.__version__} thành công từ: {src_dir}")

In [ ]:
# 3. Tạo Dense Vector Index cho CẢ 2 MODELS trên GPU
import urllib.request
from pathlib import Path
from legal_rag.retrieval.dense import build_dense_index

chunks_file = Path("/content/LegalRAG/data/chunks/legal_aware_run06/labor_code_2019_legal_aware.jsonl")

# Tự động tải từ GitHub nếu file chưa có sẵn
if not chunks_file.exists():
    chunks_file.parent.mkdir(parents=True, exist_ok=True)
    raw_url = "https://raw.githubusercontent.com/ngminhduc129/LegalRAG/main/data/chunks/legal_aware_run06/labor_code_2019_legal_aware.jsonl"
    print(f"Đang tải file chunks: {raw_url} ...")
    urllib.request.urlretrieve(raw_url, str(chunks_file))

output_dir = Path("/content/LegalRAG/data/indexes/dense")
output_dir.mkdir(parents=True, exist_ok=True)

# Danh sách 2 model embedding cần build
EMBEDDING_MODELS = [
    {
        "name": "MiniLM-L12-v2",
        "model_id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "output_path": output_dir / "dense_legal_aware_run06.pkl",
        "batch_size": 32,
    },
    {
        "name": "BGE-M3",
        "model_id": "BAAI/bge-m3",
        "output_path": output_dir / "dense_legal_aware_run06_bge_m3.pkl",
        "batch_size": 16,
    },
]

built_indexes = {}

for cfg in EMBEDDING_MODELS:
    print("\n" + "=" * 60)
    print(f"BẮT ĐẦU BUILD INDEX: {cfg['name']} ({cfg['model_id']})")
    print(f"File đích: {cfg['output_path']}")
    print("=" * 60)
    
    idx = build_dense_index(
        chunks_path=chunks_file,
        index_path=cfg["output_path"],
        model_name=cfg["model_id"],
        batch_size=cfg["batch_size"],
        max_length=256,
    )
    built_indexes[cfg["name"]] = (idx, cfg["output_path"])
    print(f"✓ Hoàn tất {cfg['name']}: {len(idx.documents)} docs | Vector shape: {idx.embeddings.shape}")

In [ ]:
# 4. So sánh kết quả truy vấn giữa MiniLM-L12-v2 và BGE-M3
test_queries = [
    "Người lao động có quyền đơn phương chấm dứt hợp đồng lao động khi nào?",
    "Thời giờ làm việc bình thường không quá bao nhiêu giờ trong một ngày?",
    "Mức lương tối thiểu được xác lập dựa trên những yếu tố nào?"
]

print("=" * 75)
print("SO SÁNH TRỰC DIỆN KẾT QUẢ RETRIEVAL (MiniLM vs BGE-M3)")
print("=" * 75)

for q_idx, query in enumerate(test_queries, 1):
    print(f"\n{'#' * 75}")
    print(f"[CÂU HỎI {q_idx}]: \"{query}\"")
    print(f"{'#' * 75}")
    
    for model_name, (idx, _) in built_indexes.items():
        print(f"\n--- Mô hình: {model_name} (Vector dim: {idx.embeddings.shape[1]}) ---")
        results = idx.search(query, k=2)
        for rank, (doc_id, score, doc) in enumerate(results, 1):
            art = doc.get("article", "N/A")
            title = doc.get("article_title", "")
            clause = doc.get("clause", "")
            prefix = f"Điều {art}"
            if clause:
                prefix += f", Khoản {clause}"
            print(f"  Top {rank} (Score: {score:.4f}): {prefix} - {title}")
            snippet = doc.get("content", "").replace("\n", " ").strip()
            print(f"    Trích đoạn: {snippet[:130]}...")

In [ ]:
# 5. Danh sách các file Index đã tạo và hướng dẫn tải về máy
print("=" * 60)
print("DANH SÁCH FILE INDEX HOÀN THÀNH:")
print("=" * 60)

for model_name, (_, path) in built_indexes.items():
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"- {model_name:15}: {path} ({size_mb:.2f} MB)")

print("\nĐể tải file về máy tính, bỏ comment dòng bên dưới:")
# from google.colab import files
# for _, (_, path) in built_indexes.items():
#     files.download(str(path))